# Week 5: Hybrid Search (BM25 + Dense + RRF)

## Goal
Implement and evaluate hybrid search to address dense-only retrieval limitations:
- **Q06** (AS281DAW 필터 수명): Model number exact match failure
- **Q09, Q18, Q22**: Explicit category keywords ignored by dense
- **Q19** (필터 교체 후): Cross-category term confusion

## Approach
1. BM25 Retriever (lexical matching)
2. Dense Retriever (C3 vectorstore)
3. Hybrid via RRF (Reciprocal Rank Fusion)

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from src.vectorstore import load_vectorstore
from src.evaluation import load_eval_questions, category_accuracy

## 1. Load C3 Baseline

In [ ]:
CHROMA_DIR = Path('../data/chroma_db_c3')
EVAL_PATH = Path('../docs/eval_questions_v2.json')

# Load vectorstore
vs = load_vectorstore(CHROMA_DIR, collection_name='lg_manuals_c3')
print(f"Loaded vectorstore with {vs._collection.count()} documents")

# Load eval questions
questions = load_eval_questions(EVAL_PATH)
print(f"Loaded {len(questions)} evaluation questions")

## 2. Extract Documents for BM25

BM25 needs access to all document texts. We'll extract from the vectorstore.

In [ ]:
# Get all documents from Chroma
all_data = vs._collection.get(include=['documents', 'metadatas'])
texts = all_data['documents']
metadatas = all_data['metadatas']

print(f"Extracted {len(texts)} documents for BM25")
print(f"Sample text (first 200 chars): {texts[0][:200]}...")

## 3. BM25 Retriever

Using LangChain's BM25Retriever with rank_bm25 backend.

### Tokenization Note
Korean BM25 is sensitive to tokenization. Starting with space-based (default).
- Limitation: "필터를" vs "필터는" won't match
- Future: Try kiwipiepy or konlpy for morpheme-based tokenization

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create LangChain documents for BM25
bm25_docs = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(texts, metadatas)
]

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(bm25_docs, k=5)
print(f"Created BM25 retriever with {len(bm25_docs)} documents")

In [ ]:
# Test BM25 on model number query (Q06)
test_query = "AS281DAW 필터 수명은 얼마나 되나요?"
bm25_results = bm25_retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Expected category: airpurifier")
print("\nBM25 Top-5:")
for i, doc in enumerate(bm25_results):
    cat = doc.metadata.get('category', 'unknown')
    print(f"  {i+1}. {cat} - {doc.page_content[:80]}...")

## 4. Dense Retriever (Baseline)

In [ ]:
# Dense retriever from vectorstore
dense_retriever = vs.as_retriever(search_kwargs={'k': 5})

# Test on same query
dense_results = dense_retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Expected category: airpurifier")
print("\nDense Top-5:")
for i, doc in enumerate(dense_results):
    cat = doc.metadata.get('category', 'unknown')
    print(f"  {i+1}. {cat} - {doc.page_content[:80]}...")

## 5. Hybrid Retriever (RRF)

Using EnsembleRetriever with Reciprocal Rank Fusion.

RRF formula: `score = sum(1 / (k + rank))` where k=60 (typical)

This combines rankings without needing score normalization.

In [ ]:
from langchain.retrievers import EnsembleRetriever

# Create hybrid retriever with equal weights
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],  # Equal weight to start
)

# Test on same query
hybrid_results = hybrid_retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Expected category: airpurifier")
print("\nHybrid (RRF) Top-5:")
for i, doc in enumerate(hybrid_results[:5]):
    cat = doc.metadata.get('category', 'unknown')
    print(f"  {i+1}. {cat} - {doc.page_content[:80]}...")

## 6. Compare All 3 on Failure Cases

Test on the 9 questions that failed with dense-only.

In [ ]:
failure_ids = ['Q03', 'Q05', 'Q06', 'Q09', 'Q10', 'Q17', 'Q18', 'Q19', 'Q22']
failure_questions = [q for q in questions if q.id in failure_ids]

print("=== Comparing retrievers on 9 failure cases ===")
print()

for q in failure_questions:
    bm25_docs = bm25_retriever.invoke(q.question)[:5]
    dense_docs = dense_retriever.invoke(q.question)[:5]
    hybrid_docs = hybrid_retriever.invoke(q.question)[:5]
    
    bm25_top1 = bm25_docs[0].metadata.get('category', 'unknown') if bm25_docs else 'none'
    dense_top1 = dense_docs[0].metadata.get('category', 'unknown') if dense_docs else 'none'
    hybrid_top1 = hybrid_docs[0].metadata.get('category', 'unknown') if hybrid_docs else 'none'
    
    # Count correct in top-5
    bm25_correct = sum(1 for d in bm25_docs if d.metadata.get('category') == q.category)
    dense_correct = sum(1 for d in dense_docs if d.metadata.get('category') == q.category)
    hybrid_correct = sum(1 for d in hybrid_docs if d.metadata.get('category') == q.category)
    
    bm25_mark = '✓' if bm25_top1 == q.category else '✗'
    dense_mark = '✓' if dense_top1 == q.category else '✗'
    hybrid_mark = '✓' if hybrid_top1 == q.category else '✗'
    
    print(f"{q.id}: {q.question[:40]}...")
    print(f"  Expected: {q.category}")
    print(f"  BM25:   {bm25_mark} top1={bm25_top1}, top5={bm25_correct}/5")
    print(f"  Dense:  {dense_mark} top1={dense_top1}, top5={dense_correct}/5")
    print(f"  Hybrid: {hybrid_mark} top1={hybrid_top1}, top5={hybrid_correct}/5")
    print()

## 7. Full Evaluation: All 23 Questions

In [ ]:
def evaluate_retriever_accuracy(retriever, questions, name="Retriever"):
    """Evaluate top-1 and top-5 category accuracy."""
    top1_correct = 0
    top5_correct = 0
    
    for q in questions:
        docs = retriever.invoke(q.question)[:5]
        categories = [d.metadata.get('category', 'unknown') for d in docs]
        
        if categories and categories[0] == q.category:
            top1_correct += 1
        if q.category in categories:
            top5_correct += 1
    
    n = len(questions)
    print(f"{name}:")
    print(f"  Top-1 Accuracy: {top1_correct/n:.1%} ({top1_correct}/{n})")
    print(f"  Top-5 Accuracy: {top5_correct/n:.1%} ({top5_correct}/{n})")
    
    return {'top1': top1_correct/n, 'top5': top5_correct/n}

print("=== Full 23-Question Evaluation ===")
print()

bm25_acc = evaluate_retriever_accuracy(bm25_retriever, questions, "BM25 Only")
print()
dense_acc = evaluate_retriever_accuracy(dense_retriever, questions, "Dense Only")
print()
hybrid_acc = evaluate_retriever_accuracy(hybrid_retriever, questions, "Hybrid (BM25+Dense)")

## 8. Summary Table

In [ ]:
print("\n=== Summary ===")
print()
print("| Retriever | Top-1 Accuracy | Top-5 Accuracy |")
print("|-----------|----------------|----------------|")
print(f"| BM25 Only | {bm25_acc['top1']:.1%} | {bm25_acc['top5']:.1%} |")
print(f"| Dense Only | {dense_acc['top1']:.1%} | {dense_acc['top5']:.1%} |")
print(f"| Hybrid (RRF) | {hybrid_acc['top1']:.1%} | {hybrid_acc['top5']:.1%} |")

## 9. Detailed Per-Question Comparison

In [ ]:
print("\n=== Per-Question Results ===")
print()
print("| ID | Question | Expected | BM25 | Dense | Hybrid |")
print("|----|-----------| ---------|------|-------|--------|")

for q in questions:
    bm25_docs = bm25_retriever.invoke(q.question)[:1]
    dense_docs = dense_retriever.invoke(q.question)[:1]
    hybrid_docs = hybrid_retriever.invoke(q.question)[:1]
    
    bm25_cat = bm25_docs[0].metadata.get('category', '?')[:3] if bm25_docs else '?'
    dense_cat = dense_docs[0].metadata.get('category', '?')[:3] if dense_docs else '?'
    hybrid_cat = hybrid_docs[0].metadata.get('category', '?')[:3] if hybrid_docs else '?'
    
    bm25_mark = '✓' if bm25_docs and bm25_docs[0].metadata.get('category') == q.category else '✗'
    dense_mark = '✓' if dense_docs and dense_docs[0].metadata.get('category') == q.category else '✗'
    hybrid_mark = '✓' if hybrid_docs and hybrid_docs[0].metadata.get('category') == q.category else '✗'
    
    question_short = q.question[:20] + '...' if len(q.question) > 20 else q.question
    expected_short = q.category[:3]
    
    print(f"| {q.id} | {question_short} | {expected_short} | {bm25_mark} | {dense_mark} | {hybrid_mark} |")

## 10. Analysis: Where Each Retriever Wins

In [ ]:
bm25_wins = []  # BM25 correct, Dense wrong
dense_wins = []  # Dense correct, BM25 wrong
both_correct = []
both_wrong = []

for q in questions:
    bm25_docs = bm25_retriever.invoke(q.question)[:1]
    dense_docs = dense_retriever.invoke(q.question)[:1]
    
    bm25_correct = bm25_docs and bm25_docs[0].metadata.get('category') == q.category
    dense_correct = dense_docs and dense_docs[0].metadata.get('category') == q.category
    
    if bm25_correct and not dense_correct:
        bm25_wins.append(q)
    elif dense_correct and not bm25_correct:
        dense_wins.append(q)
    elif bm25_correct and dense_correct:
        both_correct.append(q)
    else:
        both_wrong.append(q)

print(f"Both correct: {len(both_correct)}")
print(f"BM25 wins (BM25 correct, Dense wrong): {len(bm25_wins)}")
print(f"Dense wins (Dense correct, BM25 wrong): {len(dense_wins)}")
print(f"Both wrong: {len(both_wrong)}")

if bm25_wins:
    print("\n--- BM25 wins ---")
    for q in bm25_wins:
        print(f"  {q.id}: {q.question[:50]}... (bias: {q.retrieval_bias})")

if dense_wins:
    print("\n--- Dense wins ---")
    for q in dense_wins:
        print(f"  {q.id}: {q.question[:50]}... (bias: {q.retrieval_bias})")

if both_wrong:
    print("\n--- Both wrong ---")
    for q in both_wrong:
        print(f"  {q.id}: {q.question[:50]}... (bias: {q.retrieval_bias})")

## 11. Hybrid Benefit Analysis

Does hybrid improve over the best of BM25/Dense?

In [ ]:
hybrid_unique_wins = []  # Hybrid correct when both BM25 and Dense wrong
hybrid_losses = []  # Hybrid wrong when at least one of BM25/Dense correct

for q in questions:
    bm25_docs = bm25_retriever.invoke(q.question)[:1]
    dense_docs = dense_retriever.invoke(q.question)[:1]
    hybrid_docs = hybrid_retriever.invoke(q.question)[:1]
    
    bm25_correct = bm25_docs and bm25_docs[0].metadata.get('category') == q.category
    dense_correct = dense_docs and dense_docs[0].metadata.get('category') == q.category
    hybrid_correct = hybrid_docs and hybrid_docs[0].metadata.get('category') == q.category
    
    if hybrid_correct and not bm25_correct and not dense_correct:
        hybrid_unique_wins.append(q)
    elif not hybrid_correct and (bm25_correct or dense_correct):
        hybrid_losses.append(q)

print(f"Hybrid unique wins (neither BM25 nor Dense correct): {len(hybrid_unique_wins)}")
print(f"Hybrid losses (BM25 or Dense correct, but Hybrid wrong): {len(hybrid_losses)}")

if hybrid_unique_wins:
    print("\n--- Hybrid unique wins ---")
    for q in hybrid_unique_wins:
        print(f"  {q.id}: {q.question[:50]}...")

if hybrid_losses:
    print("\n--- Hybrid losses ---")
    for q in hybrid_losses:
        print(f"  {q.id}: {q.question[:50]}...")

## 12. Reranking with Cross-Encoder

### Why Reranking?

From our failure analysis:
- Q09: airpurifier at position 3 in Hybrid Top-5 → reranker could promote to position 1
- Q06, Q22: airpurifier not in Top-5 but in Top-10 → need larger k for first stage

### 2-Stage Approach
1. First stage: Hybrid retrieves k=20
2. Second stage: Cross-encoder reranks to top-5

### Model Choice
Using `dragonkue/bge-reranker-v2-m3-ko` (Korean-specialized)

In [ ]:
# First, check if we need to install sentence-transformers
try:
    from sentence_transformers import CrossEncoder
    print("sentence-transformers already installed")
except ImportError:
    print("Installing sentence-transformers...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import CrossEncoder
    print("Installed successfully")